# 03 · Audit playing time and build per-90 profiles

Raw event totals reflect both opportunity and behavior. We now estimate participation time and calculate rates per 90 minutes. Run after notebook 02; no extra packages are needed.

**Convention:** elapsed match-clock participation, including stoppage time and excluding halftime breaks and shootouts. Full-match players may exceed 90 minutes. Temporary Player Off/On events are counted for review but are **not subtracted**; these are registered-participation estimates, not exact seconds physically on the field. Permanent departures and dismissals end participation. Rates will differ from providers that assign exactly 90 minutes to a regulation full match.

This remains descriptive. One match per player cannot establish stable ability.


In [ ]:
import sys
import csv
import json
from collections import Counter, defaultdict
from pathlib import Path

assert sys.version_info >= (3, 11), "Use a Python 3.11+ kernel."
PROJECT_OVERRIDE = None  # Optional: Path("/path/to/soccer-recruitment")
if PROJECT_OVERRIDE is not None:
    candidates = [Path(PROJECT_OVERRIDE).expanduser()]
else:
    candidates = []
    for folder in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([folder, folder / "soccer-recruitment"])
ROOT = next((p.resolve() for p in candidates if (p / "src/soccer/cli.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory, or set PROJECT_OVERRIDE to your complete checkout.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
for folder in ("reports", "data/processed"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("Project:", ROOT.name)

paths = sorted((ROOT / "data/raw/statsbomb/events").glob("*.json"))
if not paths:
    raise FileNotFoundError("Run notebook 01 with acquisition enabled first.")
print("Downloaded event matches:", len(paths))


In [ ]:
matches = {}
for path in sorted((ROOT / "data/raw/statsbomb/matches").glob("*.json")):
    for match in json.loads(path.read_text(encoding="utf-8")):
        matches[str(match["match_id"])] = match

events_by_match = {}
for path in paths:
    events_by_match[path.stem] = json.loads(path.read_text(encoding="utf-8"))
    match = matches.get(path.stem)
    if match is None:
        raise ValueError(f"No match metadata for {path.stem}")
    print(f"{path.stem} | {match['match_date']} | "
          f"{match['competition']['competition_name']} {match['season']['season_name']} | "
          f"{match['home_team']['home_team_name']} vs {match['away_team']['away_team_name']} | "
          f"{len(events_by_match[path.stem]):,} events")
print("Total events:", sum(map(len, events_by_match.values())))


## 1. Reconstruct participation intervals

Start with Starting XI events and validate IDs against the downloaded lineup roster. Add substitutes at their substitution timestamp; stop the replaced player, dismissed player, or recorded permanent departure. Use period-relative timestamps plus actual prior-period durations. Using `minute` alone would mishandle first-half stoppage time.

Require a Half End event for every played period. Exclude period 5 (shootout). Treat temporary departure events as review flags without changing the registered-participation clock. An unsupported or inconsistent player transition raises an error instead of silently guessing minutes.


In [ ]:
def seconds(timestamp):
    h, m, s = timestamp.split(":")
    return int(h)*3600 + int(m)*60 + float(s)

def playing_time(events, roster_ids):
    played_periods = sorted({e["period"] for e in events if 1 <= e["period"] <= 4})
    if played_periods != list(range(1, max(played_periods)+1)):
        raise ValueError("Missing played period")
    durations = {}
    offsets = {}
    for period in played_periods:
        ends = [seconds(e["timestamp"]) for e in events if e["period"] == period and e["type"]["name"] == "Half End"]
        if not ends:
            raise ValueError(f"Period {period} has no Half End")
        offsets[period] = sum(durations.values())
        durations[period] = max(ends)
    total = sum(durations.values())
    active, rows, intervals = {}, {}, []
    removed = set()
    temporary_off = Counter()
    red_cards = Counter()
    permanent_off = Counter()
    bench_discipline_events = 0

    def ensure(team, player):
        key = (team["id"], player["id"])
        if key not in roster_ids:
            raise ValueError(f"Player absent from lineup roster: {key}")
        rows.setdefault(key, dict(team_id=team["id"], team=team["name"], player_id=player["id"], player=player["name"], minutes=0.0))
        return key

    def close(key, t, reason):
        if key not in active:
            raise ValueError(f"{reason}: player not active: {key}")
        start = active.pop(key)
        if t < start:
            raise ValueError("Negative participation interval")
        rows[key]["minutes"] += (t-start)/60
        intervals.append((key, start, t, reason))
        removed.add(key)

    for e in events:
        if e["type"]["name"] == "Starting XI":
            for slot in e["tactics"]["lineup"]:
                key = ensure(e["team"], slot["player"])
                if key in active:
                    raise ValueError("Duplicate starter")
                active[key] = 0.0
    teams = {key[0] for key in active}
    if len(teams) != 2 or any(sum(k[0] == team for k in active) != 11 for team in teams):
        raise ValueError("Expected exactly two Starting XIs of eleven players")
    ordered = sorted((e for e in events if e["period"] in durations), key=lambda e: (e["period"], seconds(e["timestamp"]), e.get("index", 0)))
    for e in ordered:
        t = offsets[e["period"]] + seconds(e["timestamp"])
        if seconds(e["timestamp"]) > durations[e["period"]] + 0.001:
            raise ValueError("Event occurs after period end")
        kind = e["type"]["name"]
        if kind == "Substitution":
            outgoing = ensure(e["team"], e["player"])
            # A prior permanent-departure event may already have ended this player's interval.
            if outgoing in active:
                close(outgoing, t, "substitution")
            elif outgoing not in removed:
                raise ValueError("Outgoing substitute never active")
            incoming = ensure(e["team"], e["substitution"]["replacement"])
            if incoming in active or incoming in removed:
                raise ValueError("Unsupported substitute re-entry")
            active[incoming] = t
        if kind == "Player Off":
            key = ensure(e["team"], e["player"])
            if e.get("player_off", {}).get("permanent", False):
                permanent_off[key[0]] += 1
                if key in active:
                    close(key, t, "permanent departure")
            else:
                temporary_off[key] += 1
        if kind == "Bad Behaviour" and e.get("position", {}).get("name") == "Substitute":
            bench_discipline_events += 1
            continue  # Bench discipline does not create participation time or an on-field dismissal.
        card = e.get("foul_committed", {}).get("card", {}).get("name") or e.get("bad_behaviour", {}).get("card", {}).get("name")
        if card in ("Red Card", "Second Yellow"):
            key = ensure(e["team"], e["player"])
            if key in active:
                close(key, t, card)
                red_cards[key[0]] += 1
            else:
                raise ValueError(f"Dismissal of inactive player needs manual review: {key}")
    for key in list(active):
        close(key, total, "match end")
    for key, row in rows.items():
        row["temporary_off_events"] = temporary_off[key]
        if not 0 <= row["minutes"] <= total/60 + 1e-8:
            raise ValueError("Invalid minutes")
    audit = dict(match_elapsed_minutes=total/60, period_minutes={str(p):d/60 for p,d in durations.items()},
                 temporary_off_events=sum(temporary_off.values()), dismissals=sum(red_cards.values()),
                 permanent_departures=sum(permanent_off.values()), participants=len(rows), bench_discipline_events=bench_discipline_events)
    for team in teams:
        actual = sum(r["minutes"] for k,r in rows.items() if k[0] == team)
        expected = 11*total/60
        if actual > expected + 1e-6:
            raise ValueError("Team minutes exceed eleven-player maximum")
        if not red_cards[team] and not permanent_off[team] and abs(actual-expected) > 1e-6:
            raise ValueError("Team participation time does not reconcile")
    return rows, audit


In [ ]:
minutes_by_match = {}
audits = {}
for match_id, events in events_by_match.items():
    roster_path = ROOT / f"data/raw/statsbomb/lineups/{match_id}.json"
    roster = json.loads(roster_path.read_text(encoding="utf-8"))
    roster_ids = {(team["team_id"], slot["player_id"]) for team in roster for slot in team["lineup"]}
    minutes_by_match[match_id], audits[match_id] = playing_time(events, roster_ids)
    print(match_id, json.dumps(audits[match_id], indent=2))


## 2. Attach event totals and calculate rates

For each metric: **per 90 = count / estimated participation minutes × 90**. Pass completion is a proportion and is not multiplied by 90. Pressure counts do not measure pressure success. Provider xG sums do not measure overall player value.

A minimum-minutes filter controls which rows are displayed, not which rows are saved. The illustrative 60-minute threshold is not enough evidence for recruitment decisions. These rates are also not adjusted for possession, role, opponent, score state, or pass difficulty.


In [ ]:
totals = {}
for match_id, events in events_by_match.items():
    for key, minute_row in minutes_by_match[match_id].items():
        row = totals.setdefault(key, dict(team_id=key[0], team=minute_row["team"], player_id=key[1], player=minute_row["player"],
            matches=0, minutes=0.0, passes=0, completed_passes=0, shots=0, xg=0.0, pressures=0, temporary_off_events=0))
        row["matches"] += 1
        row["minutes"] += minute_row["minutes"]
        row["temporary_off_events"] += minute_row["temporary_off_events"]
    for e in events:
        if e["period"] == 5 or "player" not in e:
            continue
        kind = e["type"]["name"]
        if kind not in ("Pass", "Shot", "Pressure"):
            continue
        key = (e["team"]["id"], e["player"]["id"])
        if key not in minutes_by_match[match_id]:
            raise ValueError(f"Action player has no participation interval: {match_id}, {key}")
        row = totals[key]
        if kind == "Pass":
            row["passes"] += 1
            row["completed_passes"] += int("outcome" not in e["pass"])
        elif kind == "Shot":
            row["shots"] += 1
            row["xg"] += e["shot"].get("statsbomb_xg", 0.0)
        else:
            row["pressures"] += 1

profiles = []
for key, row in sorted(totals.items()):
    for metric in ("passes", "completed_passes", "shots", "xg", "pressures"):
        row[f"{metric}_per90"] = row[metric]*90/row["minutes"] if row["minutes"] > 0 else None
    row["pass_completion_pct"] = 100*row["completed_passes"]/row["passes"] if row["passes"] else None
    profiles.append(row)
MIN_MINUTES = 60
print(f"Showing first 15 players alphabetically with at least {MIN_MINUTES} estimated minutes.")
for row in sorted((r for r in profiles if r["minutes"] >= MIN_MINUTES), key=lambda r: (r["team"],r["player"]))[:15]:
    print(f"{row['player'][:30]:30s} | minutes {row['minutes']:6.1f} | passes/90 {row['passes_per90']:6.1f} | pressures/90 {row['pressures_per90']:5.1f}")


## 3. Validate and export

Reconcile event totals, confirm the rate identity, and check that no half-time break was added. Team totals were checked above: eleven-player elapsed participation is the maximum; without recorded dismissals or permanent departures, substitution transitions must preserve it.

The parser deliberately stops on unsupported transitions. Red cards, permanent departures, abandoned games and unusual formats require manual review when expanding the sample. Temporary injury absences remain included under the stated exposure convention. This is not a substitute for audited official playing-time records.


In [ ]:
import math
for kind, metric in [("Pass","passes"),("Shot","shots"),("Pressure","pressures")]:
    expected = sum(e["type"]["name"] == kind and "player" in e and e["period"] != 5 for events in events_by_match.values() for e in events)
    assert sum(r[metric] for r in profiles) == expected
for row in profiles:
    if row["minutes"] > 0:
        assert math.isclose(row["passes_per90"]*row["minutes"]/90, row["passes"])
assert math.isclose(seconds("00:59:03.591"), 3543.591)
for audit in audits.values():
    assert math.isclose(sum(audit["period_minutes"].values()), audit["match_elapsed_minutes"])

output = ROOT / "reports/player_per90_profiles.csv"
with output.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(profiles[0]))
    writer.writeheader()
    writer.writerows(profiles)
(ROOT / "reports/minutes_audit.json").write_text(json.dumps(audits, indent=2)+"\n", encoding="utf-8")
print("Checks passed. Saved:", output)


## What to notice

England–Iran includes substantial stoppage time. A full-match player's elapsed participation can exceed 110 minutes under our convention. That is not an error and illustrates why exposure definitions must be documented. Their per-90 total will therefore be lower than a convention that assigns exactly 90 minutes.

We have improved exposure adjustment, but the sample still has one match per player. Do not identify the best recruit by sorting these rates. Next, choose a target role and expand suitable event coverage, then construct contextual features and uncertainty estimates. Keep historical descriptive profiles separate from a prospective model: only matches before the scouting date belong in its features.
